In [1]:
import pandas as pd
import sqlite3
import os

#Loading CSVs
branches_df = pd.read_csv('data/branches.csv')
products_df = pd.read_csv('data/products.csv')
customers_df = pd.read_csv('data/customers.csv')
orders_df = pd.read_csv('data/orders.csv')

print("CSVs loaded:")
print(f"  branches:  {len(branches_df)} rows")
print(f"  products:  {len(products_df)} rows")
print(f"  customers: {len(customers_df)} rows")
print(f"  orders:    {len(orders_df)} rows")

CSVs loaded:
  branches:  10 rows
  products:  120 rows
  customers: 200 rows
  orders:    5000 rows


In [3]:
#Creating database
conn = sqlite3.connect('data/automotive_nz.db')

#Load each df into the db as table
branches_df.to_sql('branches', conn, if_exists='replace', index=False)
products_df.to_sql('products', conn, if_exists='replace', index=False)
customers_df.to_sql('customers', conn, if_exists='replace', index=False)
orders_df.to_sql('orders', conn, if_exists='replace', index=False)

print("Database crsated: data/automotive_nz.db")
print("\n Tables loaded:")

#veryfy the database by querying
for table in ['branches', 'products', 'customers', 'orders']:
    count = pd.read_sql(f"SELECT COUNT(*) as rows FROM {table}", conn).iloc[0]['rows']
    print(f" {table}: {count} rows")

Database crsated: data/automotive_nz.db

 Tables loaded:
 branches: 10 rows
 products: 120 rows
 customers: 200 rows
 orders: 5000 rows


In [4]:
def run_query(sql, description=""):
    if description:
        print(f"\n--- {description} ---")
    result = pd.read_sql(sql, conn)
    print(result.to_string(index=False))
    return result

In [8]:
#Business Question 1: Which branches generate the most revenue?

run_query("""
          SELECT 
              b.branch_id, 
              b.branch_name, 
              b.brand, 
              b.region, 
              b.segment,
              COUNT(o.order_id) AS total_orders,
              SUM(o.revenue) AS total_revenue,
              AVG(o.revenue) AS average_order_value
          FROM orders o
          INNER JOIN branches b ON b.branch_id = o.branch_id
          GROUP BY b.branch_id, b.branch_name, b.brand, b.region, b.segment
          ORDER BY total_revenue DESC
          """, "Branch Revenue Ranking")


--- Branch Revenue Ranking ---
 branch_id               branch_name        brand       region              segment  total_orders  total_revenue  average_order_value
         9   Diesel Distributors AKL  Diesel Dist     Auckland Specialist Wholesale           530      180632.44           340.815925
         8     Battery Town Hamilton Battery Town     Hamilton              Service           517      178325.66           344.923907
         6 HCB Technologies Auckland          HCB     Auckland Specialist Wholesale           522      165037.73           316.164234
         4         Autolign Auckland     Autolign     Auckland Specialist Wholesale           500      161781.73           323.563460
         7     Battery Town Auckland Battery Town     Auckland              Service           492      160425.15           326.067378
        10              BNT Hamilton          BNT     Hamilton                Trade           492      159741.25           324.677337
         3            BNT Well

,branch_id,branch_name,brand,region,segment,total_orders,total_revenue,average_order_value
0,9,Diesel Distributors AKL,Diesel Dist,Auckland,Specialist Wholesale,530,180632.44,340.815925
1,8,Battery Town Hamilton,Battery Town,Hamilton,Service,517,178325.66,344.923907
2,6,HCB Technologies Auckland,HCB,Auckland,Specialist Wholesale,522,165037.73,316.164234
3,4,Autolign Auckland,Autolign,Auckland,Specialist Wholesale,500,161781.73,323.563460
4,7,Battery Town Auckland,Battery Town,Auckland,Service,492,160425.15,326.067378
5,10,BNT Hamilton,BNT,Hamilton,Trade,492,159741.25,324.677337
6,3,BNT Wellington,BNT,Wellington,Trade,495,159615.78,322.456121
7,2,BNT Christchurch,BNT,Christchurch,Trade,496,157344.58,317.226976
8,5,Autolign Wellington,Autolign,Wellington,Specialist Wholesale,470,154268.02,328.229830
9,1,BNT Auckland Central,BNT,Auckland,Trade,486,150632.98,309.944403


In [12]:
#Business Question 2: Revenue by segment

run_query("""
    SELECT 
        b.segment,
        COUNT(DISTINCT b.branch_id) AS number_of_branches,
        COUNT(o.order_id) AS total_orders,
        ROUND(SUM(o.revenue),2) AS total_revenue,
        ROUND(AVG(o.revenue),2) AS avg_order_value,
        ROUND(SUM(o.revenue)*100/(SELECT SUM(revenue) FROM orders),1) AS revenue_share_pct
    FROM orders o
    INNER JOIN branches b ON b.branch_id = o.branch_id
    GROUP BY b.segment
    ORDER BY total_revenue DESC
""", "Revenue by segment")



--- Revenue by segment ---
             segment  number_of_branches  total_orders  total_revenue  avg_order_value  revenue_share_pct
Specialist Wholesale                   4          2022      661719.92           327.26               40.7
               Trade                   4          1969      627334.59           318.61               38.5
             Service                   2          1009      338750.81           335.73               20.8


,segment,number_of_branches,total_orders,total_revenue,avg_order_value,revenue_share_pct
0,Specialist Wholesale,4,2022,661719.92,327.26,40.7
1,Trade,4,1969,627334.59,318.61,38.5
2,Service,2,1009,338750.81,335.73,20.8


In [13]:
#Business Question 3: Seasonal trends

run_query("""
    SELECT
        CASE CAST(strftime('%m', order_date) AS INTEGER)
            WHEN 1  THEN '01-Jan' WHEN 2  THEN '02-Feb'
            WHEN 3  THEN '03-Mar' WHEN 4  THEN '04-Apr'
            WHEN 5  THEN '05-May' WHEN 6  THEN '06-Jun'
            WHEN 7  THEN '07-Jul' WHEN 8  THEN '08-Aug'
            WHEN 9  THEN '09-Sep' WHEN 10 THEN '10-Oct'
            WHEN 11 THEN '11-Nov' WHEN 12 THEN '12-Dec'
        END                                    AS month,
        COUNT(o.order_id)                      AS total_orders,
        ROUND(SUM(o.revenue), 2)               AS total_revenue,
        ROUND(SUM(CASE WHEN p.category = 'Batteries' 
                  THEN o.revenue ELSE 0 END), 2) AS battery_revenue,
        ROUND(SUM(CASE WHEN p.category = 'Filters'   
                  THEN o.revenue ELSE 0 END), 2) AS filter_revenue,
        ROUND(SUM(CASE WHEN p.category = 'Suspension' 
                  THEN o.revenue ELSE 0 END), 2) AS suspension_revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY strftime('%m', order_date)
    ORDER BY strftime('%m', order_date)
""", "Monthly seasonal trends")


--- Monthly seasonal trends ---
 month  total_orders  total_revenue  battery_revenue  filter_revenue  suspension_revenue
01-Jan           439      116906.79         31604.53        11754.15            27076.67
02-Feb           380      113821.72         32441.84         9218.82            31841.78
03-Mar           439      129383.74         31560.16         7134.56            43415.84
04-Apr           399      128779.32         35625.81         7356.90            39069.24
05-May           411      136955.63         32549.16         7067.70            46013.13
06-Jun           423      153836.26         70619.72         4936.23            35991.11
07-Jul           405      154497.60         69051.85         2944.56            43269.46
08-Aug           421      165828.34         81912.18         3672.34            39616.02
09-Sep           446      134880.77         24429.77         8805.38            50508.12
10-Oct           442      149024.89         33632.60         7084.88         

,month,total_orders,total_revenue,battery_revenue,filter_revenue,suspension_revenue
0,01-Jan,439,116906.79,31604.53,11754.15,27076.67
1,02-Feb,380,113821.72,32441.84,9218.82,31841.78
2,03-Mar,439,129383.74,31560.16,7134.56,43415.84
3,04-Apr,399,128779.32,35625.81,7356.90,39069.24
4,05-May,411,136955.63,32549.16,7067.70,46013.13
5,06-Jun,423,153836.26,70619.72,4936.23,35991.11
6,07-Jul,405,154497.60,69051.85,2944.56,43269.46
7,08-Aug,421,165828.34,81912.18,3672.34,39616.02
8,09-Sep,446,134880.77,24429.77,8805.38,50508.12
9,10-Oct,442,149024.89,33632.60,7084.88,51321.89


In [14]:
queries = {
    "01_branch_revenue.sql": """
SELECT 
    b.branch_name,
    b.brand,
    b.region,
    b.segment,
    COUNT(o.order_id)          AS total_orders,
    ROUND(SUM(o.revenue), 2)   AS total_revenue,
    ROUND(AVG(o.revenue), 2)   AS avg_order_value
FROM orders o
JOIN branches b ON o.branch_id = b.branch_id
GROUP BY b.branch_id, b.branch_name, b.brand, b.region, b.segment
ORDER BY total_revenue DESC;
""",

    "02_segment_revenue.sql": """
SELECT
    b.segment,
    COUNT(DISTINCT b.branch_id)  AS number_of_branches,
    COUNT(o.order_id)            AS total_orders,
    ROUND(SUM(o.revenue), 2)     AS total_revenue,
    ROUND(AVG(o.revenue), 2)     AS avg_order_value,
    ROUND(SUM(o.revenue) * 100.0 / 
          (SELECT SUM(revenue) FROM orders), 1) AS revenue_share_pct
FROM orders o
JOIN branches b ON o.branch_id = b.branch_id
GROUP BY b.segment
ORDER BY total_revenue DESC;
""",

    "03_seasonal_trends.sql": """
SELECT
    CASE CAST(strftime('%m', order_date) AS INTEGER)
        WHEN 1  THEN '01-Jan' WHEN 2  THEN '02-Feb'
        WHEN 3  THEN '03-Mar' WHEN 4  THEN '04-Apr'
        WHEN 5  THEN '05-May' WHEN 6  THEN '06-Jun'
        WHEN 7  THEN '07-Jul' WHEN 8  THEN '08-Aug'
        WHEN 9  THEN '09-Sep' WHEN 10 THEN '10-Oct'
        WHEN 11 THEN '11-Nov' WHEN 12 THEN '12-Dec'
    END                                      AS month,
    COUNT(o.order_id)                        AS total_orders,
    ROUND(SUM(o.revenue), 2)                 AS total_revenue,
    ROUND(SUM(CASE WHEN p.category = 'Batteries'
              THEN o.revenue ELSE 0 END), 2) AS battery_revenue,
    ROUND(SUM(CASE WHEN p.category = 'Filters'
              THEN o.revenue ELSE 0 END), 2) AS filter_revenue,
    ROUND(SUM(CASE WHEN p.category = 'Suspension'
              THEN o.revenue ELSE 0 END), 2) AS suspension_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY strftime('%m', order_date)
ORDER BY strftime('%m', order_date);
"""
}

os.makedirs('sql', exist_ok=True)
for filename, query in queries.items():
    with open(f'sql/{filename}', 'w') as f:
        f.write(query.strip())
    print(f"Saved: sql/{filename}")

print("\nAll queries saved.")

Saved: sql/01_branch_revenue.sql
Saved: sql/02_segment_revenue.sql
Saved: sql/03_seasonal_trends.sql

All queries saved.
